# 12 — Play and watch AHR Go

Play a **9×9 game** against a selected AHR checkpoint, or advance an AHR-versus-AHR game. Click an intersection, pass, undo one move, or start a new game. Watch mode runs only a bounded number of moves when you click its button; Run All never starts a long game.

**Inputs:** tagged definition cells from notebooks 00–06, 08 and 10; a trained 08/09 checkpoint, explicitly selected below. Default: `go08-initial-warmup-v2`, best checkpoint. **Outputs:** a live notebook widget, a portable board image, validation receipt, and a resumable game under `checkpoints/<session_id>/`. **Runtime:** normally under 2 minutes for Run All on CPU; each requested AI move uses the selected search budget plus verification overhead. No network service is published.

This is an interface to a candidate model. Playing strength is unmeasured. The rules use positional superko and area scoring with komi 7.5; dead stones must be captured before both players pass. Search cutoffs retain **UNKNOWN** strategic status. A bounded watch batch ending is a pause, never a scored loss.

### Resume exactly
Restart the `ahr-go` kernel and Run All with the **same session ID, model run, selection, search settings and device**. The game automatically restores the latest hash-verified checkpoint, full superko history, undo timeline and random states. Its original model payload is copied locally, so later upstream training/checkpoint retention cannot change the opponent. Changed or corrupt manifests are rejected; no silent reset occurs. Every move, undo, new game and setting change is saved atomically. The latest two valid game snapshots are retained; `best` identifies the latest saved game, not a strength judgment. Only one writer may commit a given revision.

To play a later notebook 09 candidate, choose its run ID and a **new session ID**. CPU is the default here so GPU training can continue independently. In JupyterLab open this notebook with the `ahr-go` kernel; if widget controls are unavailable, the Python session API and static board image remain usable.

Version 2 defaults to the completed `go09-first-selfplay-v1` candidate, selection `latest`, with a new `play12-v2-main` session. Existing games are preserved and require their original source snapshot; source changes never silently rewrite a saved game. The end-to-end companion `reports/12_v2_complete_games.ipynb` checks both human colors using an explicitly scripted legal human fixture, full-game save/resume, and end-to-end response time. It does not claim that a human has reviewed the browser controls.

In [ ]:
from pathlib import Path
import os, json, nbformat
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/00_instance_setup_and_recovery.ipynb").exists())
for number in (*range(7), 8, 10):
    path = next((PROJECT / "notebooks").glob(f"{number:02d}_*.ipynb"))
    for index, cell in enumerate(nbformat.read(path, 4).cells):
        if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", []):
            exec(compile(cell.source, f"{path.name}:export-{index}", "exec"), globals())
import torch
import ipywidgets as widgets
from IPython.display import display, HTML, Image
# Inference only; no training or upstream workflow cells are executed.
torch.set_num_threads(1)
print("Ready: notebook controls, exact rules, and AHR definitions loaded.")

## 1. Saved-game integrity

These reusable definitions pin the model and source versions, replay-check every saved move against two rule implementations, and reject concurrent or stale writers. Game snapshots contain no training optimizer because this notebook performs inference only; the copied original trained payload retains its training recovery state. All initialized RNG generators are included.

In [ ]:
import base64, copy, hashlib, json, math, os, random, re, shutil, time, uuid, fcntl
from contextlib import contextmanager
from pathlib import Path
import numpy as np

PLAY12_SCHEMA = "ahr12-play-session-v1"
PLAY12_RESUME = "12: restart ahr-go kernel and Run All with identical SESSION12, MODEL_RUN12, SELECTION12, DEVICE12 and SEARCH12"

def play12_sources(project):
    result = {}
    for number in (*range(7), 8, 10, 12):
        path = next((Path(project) / "notebooks").glob(f"{number:02d}_*.ipynb"))
        text = "\n".join(c.source for c in nbformat.read(path, 4).cells if c.cell_type == "code" and "ahr-export" in c.metadata.get("tags", []))
        result[path.name] = hashlib.sha256(text.encode()).hexdigest()
    return result

def play12_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)

def play12_rng():
    np_state = np.random.get_state()
    return {"python": random.getstate(), "numpy": [np_state[0], np_state[1].tolist(), *np_state[2:]],
            "torch_cpu": torch.get_rng_state().tolist(),
            "torch_cuda": [v.tolist() for v in torch.cuda.get_rng_state_all()] if torch.cuda.is_initialized() else None}

def play12_restore_rng(value):
    def tuples(v): return tuple(tuples(x) for x in v) if isinstance(v, list) else v
    random.setstate(tuples(value["python"]))
    name, keys, position, has_gauss, cached = value["numpy"]
    np.random.set_state((name, np.asarray(keys, dtype=np.uint32), position, has_gauss, cached))
    torch.set_rng_state(torch.tensor(value["torch_cpu"], dtype=torch.uint8))
    if value["torch_cuda"] is not None:
        if not torch.cuda.is_available() or len(value["torch_cuda"]) != torch.cuda.device_count():
            raise ValueError("Saved CUDA RNG topology differs; explicit migration required")
        torch.cuda.set_rng_state_all([torch.tensor(v, dtype=torch.uint8) for v in value["torch_cuda"]])

@contextmanager
def play12_lock(root):
    root.mkdir(parents=True, exist_ok=True)
    with open(root / ".writer.lock", "a+") as handle:
        try: fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error: raise RuntimeError("Another writer owns this game session") from error
        try: yield
        finally: fcntl.flock(handle.fileno(), fcntl.LOCK_UN)

def play12_validate_game(payload):
    if payload.get("schema") != PLAY12_SCHEMA or type(payload.get("revision")) is not int or payload["revision"] < 0:
        raise ValueError("Invalid saved-game schema/revision")
    if payload.get("mode") not in ("black", "white", "watch") or not 0 < payload.get("time_budget_s", 0) <= 10:
        raise ValueError("Invalid saved-game settings")
    timeline = payload["timeline"]
    if not timeline: raise ValueError("Empty game timeline")
    state = GoState.from_dict(timeline[0]["state"])
    if state.to_dict() != initial_state(9, payload["komi"]).to_dict() or timeline[0].get("action") is not None:
        raise ValueError("Saved game must begin at its exact empty-board state")
    for row in timeline[1:]:
        candidate = play(state, row["action"])
        independent = independent_go_successor(state, row["action"])
        recorded = GoState.from_dict(row["state"])
        if independent is None or candidate.to_dict() != independent.to_dict() or candidate.to_dict() != recorded.to_dict():
            raise ValueError("Saved game disagrees with independent replay")
        state = candidate
    if state.to_dict() != payload["state"]: raise ValueError("Saved latest state differs from timeline")
    return state

def play12_read(root, project):
    root = Path(root); path = root / "manifest.json"
    if not path.exists():
        if list(root.glob("game-*.json")) or (root / "model.pt").exists():
            raise ValueError("Game payload exists without a manifest; refusing reset")
        return None
    manifest = json.loads(path.read_text())
    entries = manifest["entries"]
    if manifest.get("schema") != PLAY12_SCHEMA or not entries or len({e["file"] for e in entries}) != len(entries):
        raise ValueError("Invalid game manifest")
    if len({e["revision"] for e in entries}) != len(entries) or manifest["latest"] != max(entries, key=lambda e:e["revision"])["file"]:
        raise ValueError("Game manifest revision would rewind")
    if manifest["best"] not in {e["file"] for e in entries}: raise ValueError("Missing best game snapshot")
    states = {}
    for entry in entries:
        name = entry["file"]
        if re.fullmatch(r"game-[0-9]{8}\.json", name) is None: raise ValueError("Unsafe game checkpoint filename")
        payload_path = root / name
        if file_sha256(payload_path) != entry["sha256"]: raise ValueError("Corrupt saved game")
        payload = json.loads(payload_path.read_text())
        if payload["revision"] != entry["revision"]: raise ValueError("Game revision mismatch")
        if payload["source_hashes"] != play12_sources(project): raise ValueError("Game source versions changed; explicit migration required")
        if payload["session_id"] != root.name: raise ValueError("Wrong session identity")
        if file_sha256(root / "model.pt") != payload["model_metadata"]["checkpoint_sha256"]:
            raise ValueError("Pinned model changed")
        play12_validate_game(payload)
        states[name] = payload
    return states[manifest["latest"]]


## 2. Game controller

`PlaySession12` exposes `play_vertex`, `ai_move`, `undo`, `new_game` and `set_settings`. All moves are verified before saving. Two passes produce an area result; unfinished positions display no final result. Undo restores the complete earlier position, including its exact superko history.

In [ ]:
class PlaySession12:
    def __init__(self, project, session_id, model_run="go09-first-selfplay-v1", selection="latest", device="cpu", search_config=None):
        self.project = Path(project).resolve()
        if not isinstance(session_id, str) or re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,95}", session_id) is None:
            raise ValueError("Use a simple session ID")
        self.root = self.project / "checkpoints" / session_id
        self.search_config = search_config or {"max_depth": 2, "branch_width": 4, "max_nodes": 32, "queen_cycles": 1, "temperature": 1.0}
        self.search = GoAHRSearchConfig(**self.search_config)
        self.device = str(device)
        request = {"model_run": model_run, "selection": selection, "device": self.device, "search_config": self.search_config}
        with play12_lock(self.root):
            payload = play12_read(self.root, self.project)
            if payload is None:
                model, metadata = load_go_ahr(self.project, model_run, device=self.device, selection=selection)
                if model.config.board_size != 9: raise ValueError("Play interface requires a 9x9 model")
                atomic_write_bytes(self.root / "model.pt", (self.project / metadata["checkpoint_file"]).read_bytes())
                if file_sha256(self.root / "model.pt") != metadata["checkpoint_sha256"]: raise ValueError("Model copy hash mismatch")
                state = initial_state(9, 7.5)
                self.model = model
                self.payload = {"schema": PLAY12_SCHEMA, "session_id": session_id, "revision": 0,
                    "source_hashes": play12_sources(self.project), "request": request, "model_metadata": metadata,
                    "komi": 7.5, "mode": "black", "time_budget_s": 0.5, "game_number": 1,
                    "state": state.to_dict(), "timeline": [{"action": None, "state": state.to_dict(), "actor": "initial"}],
                    "last_search": None, "last_event": "new_session", "rng": play12_rng()}
                self._write_locked()
            else:
                if payload["request"] != request: raise ValueError("Resume settings differ; use the saved settings or a new session ID")
                original = torch.load(self.root / "model.pt", map_location="cpu", weights_only=False)
                if original["source_hashes"] != go08_source_hashes(self.project): raise ValueError("Pinned model architecture changed")
                self.model = GoAHR(GoAHRConfig(**original["model_config"]))
                self.model.load_state_dict(original["model"], strict=True)
                self.model.to(self.device).eval()
                if go_model_digest(self.model) != payload["model_metadata"]["model_sha256"]: raise ValueError("Loaded model digest differs")
                self.payload = payload
                play12_restore_rng(payload["rng"])
        self.state = GoState.from_dict(self.payload["state"])
        self.resumed = payload is not None

    def _write_locked(self):
        play12_validate_game(self.payload)
        revision = self.payload["revision"]
        name = f"game-{revision:08d}.json"
        atomic_json(self.root / name, self.payload)
        entry = {"file": name, "revision": revision, "sha256": file_sha256(self.root / name)}
        old_path = self.root / "manifest.json"
        previous = json.loads(old_path.read_text())["entries"] if old_path.exists() else []
        entries = sorted([e for e in previous if e["revision"] != revision] + [entry], key=lambda e:e["revision"])[-2:]
        atomic_json(old_path, {"schema": PLAY12_SCHEMA, "entries": entries, "latest": name, "best": name,
            "best_meaning": "latest user game, not a strength ranking", "resume_cell": PLAY12_RESUME})
        keep = {e["file"] for e in entries}
        for stale in previous:
            if stale["file"] not in keep: (self.root / stale["file"]).unlink(missing_ok=True)

    def _commit(self, changed, event):
        with play12_lock(self.root):
            current = play12_read(self.root, self.project)
            if current is None or current["revision"] != self.payload["revision"]:
                raise RuntimeError("This interface is stale; resume the latest game before writing")
            changed["revision"] = self.payload["revision"] + 1
            changed["last_event"] = event
            changed["rng"] = play12_rng()
            prior = self.payload
            self.payload = changed
            try: self._write_locked()
            except Exception:
                self.payload = prior
                raise
            self.state = GoState.from_dict(changed["state"])
        return self.state

    def move(self, action, actor="human", search=None):
        next_state = play(self.state, action)
        independent = independent_go_successor(self.state, action)
        if independent is None or independent.to_dict() != next_state.to_dict(): raise RuntimeError("Move failed independent verification")
        changed = copy.deepcopy(self.payload)
        changed["state"] = next_state.to_dict()
        changed["timeline"].append({"action": action, "actor": actor, "state": next_state.to_dict()})
        changed["last_search"] = search
        return self._commit(changed, actor + ":" + gtp_action_to_vertex(action, 9))

    def play_vertex(self, vertex):
        if self.payload["mode"] == "watch": raise ValueError("Switch to a human play mode first")
        human = BLACK if self.payload["mode"] == "black" else WHITE
        if self.state.to_play != human: raise ValueError("It is AHR's turn; click AHR move")
        return self.move(gtp_vertex_to_action(vertex, 9), "human")

    def ai_move(self):
        if self.state.terminal: raise ValueError("The game ended after two passes")
        interaction_started = time.perf_counter()
        result = choose_go_ahr_move(self.state, self.model, time_budget_s=self.payload["time_budget_s"], search_config=self.search)
        search = {k: result.get(k) for k in ("status", "strategic_status", "elapsed_s", "nodes", "model_calls", "deadline_missed", "fallback_pass", "reason")}
        saving_started = time.perf_counter()
        self.move(result["action"], "AHR", search)
        result = dict(result, interaction_elapsed_s=time.perf_counter()-interaction_started,
                      save_elapsed_s=time.perf_counter()-saving_started)
        return result

    def undo(self):
        if len(self.payload["timeline"]) <= 1: raise ValueError("No move to undo")
        changed = copy.deepcopy(self.payload)
        changed["timeline"].pop()
        changed["state"] = changed["timeline"][-1]["state"]
        changed["last_search"] = None
        return self._commit(changed, "undo_one_move")

    def new_game(self):
        changed = copy.deepcopy(self.payload); state = initial_state(9, changed["komi"])
        changed.update(state=state.to_dict(), timeline=[{"action": None, "actor": "initial", "state": state.to_dict()}],
                       last_search=None, game_number=changed["game_number"] + 1)
        return self._commit(changed, "new_game")

    def set_settings(self, mode=None, time_budget_s=None):
        changed = copy.deepcopy(self.payload)
        if mode is not None:
            if mode not in ("black", "white", "watch"): raise ValueError("Invalid play mode")
            changed["mode"] = mode
        if time_budget_s is not None:
            if type(time_budget_s) not in (int, float) or not math.isfinite(time_budget_s) or not 0 < time_budget_s <= 10:
                raise ValueError("Move budget must be positive and at most 10 seconds")
            changed["time_budget_s"] = float(time_budget_s)
        return self._commit(changed, "settings")

    def result_text(self):
        if not self.state.terminal: return "In progress — result unknown"
        score = area_score(self.state)
        return "Draw" if score == 0 else f"{'Black' if score > 0 else 'White'} wins by {abs(score):g} (area scoring)"


## 3. Board and notebook controls

The board buttons use Go coordinates (the letter I is skipped). In human mode, clicking a legal point plays your move and requests one AHR reply. **AHR move** advances one move when it is the computer's turn; **Watch batch** advances up to the chosen count in watch mode. **Pause** takes effect between AI moves. Changing settings, undoing, or starting a new game pauses a watch batch. A saved static board below remains visible without a widget-capable viewer.

In [ ]:
import asyncio, html
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

def play12_board_figure(session, title="AHR Go · 9×9"):
    state = session.state
    fig, ax = plt.subplots(figsize=(6.1, 6.5), dpi=140)
    fig.patch.set_facecolor("#f6f4ef"); ax.set_facecolor("#ddb77b")
    for p in range(9):
        ax.plot([0,8], [p,p], color="#684820", linewidth=.8, zorder=1)
        ax.plot([p,p], [0,8], color="#684820", linewidth=.8, zorder=1)
    for x,y in ((2,2),(6,2),(4,4),(2,6),(6,6)): ax.plot(x,y,"o",color="#684820",markersize=4,zorder=2)
    for action,color in enumerate(state.board):
        if color:
            y,x=divmod(action,9)
            ax.add_patch(Circle((x,y),.43,facecolor="#171b20" if color==BLACK else "#fffefa",edgecolor="#24282b",linewidth=1.2,zorder=3))
    latest = session.payload["timeline"][-1]["action"]
    if latest is not None and latest < 81:
        y,x=divmod(latest,9); ax.add_patch(Circle((x,y),.12,fill=False,edgecolor="#dc5141",linewidth=1.8,zorder=4))
    ax.set_xticks(range(9),list(GTP_COLUMNS[:9]));ax.set_yticks(range(9),list(range(9,0,-1)))
    ax.tick_params(top=True,labeltop=True,bottom=True,labelbottom=True,length=0,pad=8,labelsize=11)
    ax.set_xlim(-.6,8.6);ax.set_ylim(8.6,-.6);ax.set_aspect("equal")
    for spine in ax.spines.values():spine.set_color("#c7a471")
    ax.set_title(title,pad=34,fontsize=17,fontweight="bold",color="#202c35")
    to_play = "Black" if state.to_play == BLACK else "White"
    fig.text(.5,.055,f"Move {state.move_number} · {to_play} to play · Komi {state.komi:g}",ha="center",fontsize=11,color="#34434b")
    fig.text(.5,.025,session.result_text(),ha="center",fontsize=10,color="#56646c")
    fig.subplots_adjust(left=.1,right=.9,top=.84,bottom=.12)
    return fig

class PlayWidget12:
    def __init__(self, session):
        self.session=session;self.task=None;self.paused=True;self.busy=False;self._sync=False
        self.title=widgets.HTML("<h2 style='margin:0'>AHR Go <span style='font-size:16px;color:#65717a'>9×9</span></h2>")
        self.status=widgets.HTML();self.message=widgets.HTML()
        self.mode=widgets.Dropdown(options=[("Play Black","black"),("Play White","white"),("Watch AHR vs AHR","watch")],value=session.payload["mode"],description="Mode:")
        self.budget=widgets.FloatSlider(min=.05,max=2,step=.05,value=min(2,session.payload["time_budget_s"]),description="Seconds:",readout_format=".2f",continuous_update=False)
        self.points=[];children=[widgets.HTML("")]+[widgets.HTML(f"<b style='display:block;text-align:center'>{x}</b>") for x in GTP_COLUMNS[:9]]
        for row in range(9):
            children.append(widgets.HTML(f"<b style='display:block;text-align:center;padding-top:7px'>{9-row}</b>"))
            for col in range(9):
                action=row*9+col
                button=widgets.Button(description="+",tooltip=gtp_action_to_vertex(action,9),layout=widgets.Layout(width="38px",height="38px",padding="0",margin="0",border="1px solid #d3ad71"))
                button.style.button_color="#ddb77b";button.style.font_size="25px";button.style.text_color="#755729"
                button.on_click(lambda _,a=action:self._human(a))
                self.points.append(button);children.append(button)
        self.board=widgets.GridBox(children,layout=widgets.Layout(grid_template_columns="26px repeat(9, 38px)",grid_template_rows="24px repeat(9, 38px)",grid_gap="0px"))
        self.pass_button=widgets.Button(description="Pass",icon="forward",layout=widgets.Layout(width="100px"))
        self.ai_button=widgets.Button(description="AHR move",icon="play",button_style="primary",layout=widgets.Layout(width="125px"))
        self.undo_button=widgets.Button(description="Undo move",icon="undo",layout=widgets.Layout(width="125px"))
        self.new_button=widgets.Button(description="New game",icon="refresh",layout=widgets.Layout(width="115px"))
        self.watch_count=widgets.BoundedIntText(value=10,min=1,max=100,description="Moves:",layout=widgets.Layout(width="165px"))
        self.watch_button=widgets.Button(description="Watch batch",icon="play",layout=widgets.Layout(width="130px"))
        self.pause_button=widgets.Button(description="Pause",icon="pause",layout=widgets.Layout(width="95px"))
        self.pass_button.on_click(lambda _:self._human(81));self.ai_button.on_click(lambda _:self._single_ai())
        self.undo_button.on_click(lambda _:self._change(session.undo));self.new_button.on_click(lambda _:self._change(session.new_game))
        self.watch_button.on_click(lambda _:self._start_watch());self.pause_button.on_click(lambda _:self.pause())
        self.mode.observe(self._settings,names="value");self.budget.observe(self._settings,names="value")
        metadata=session.payload["model_metadata"]
        caption=widgets.HTML(f"<small>Candidate: <b>{html.escape(metadata['run_id'])}</b>, step {metadata['optimizer_step']}. No playing-strength rating is assigned; evaluation results are in notebook 11.<br>Every action is saved. Click a point to play; two passes end the game.</small>")
        self.widget=widgets.VBox([self.title,caption,self.mode,self.budget,self.status,self.board,
            widgets.HBox([self.pass_button,self.ai_button,self.undo_button]),
            widgets.HBox([self.new_button,self.watch_count]),widgets.HBox([self.watch_button,self.pause_button]),self.message],
            layout=widgets.Layout(width="520px",padding="16px",border="1px solid #d7dedf"))
        self.refresh()

    def refresh(self):
        state=self.session.state;mode=self.session.payload["mode"];human=BLACK if mode=="black" else WHITE
        human_turn=mode!="watch" and state.to_play==human and not state.terminal
        legal=set(legal_actions(state)) if human_turn and not self.busy else set()
        latest=self.session.payload["timeline"][-1]["action"]
        for action,button in enumerate(self.points):
            color=state.board[action];button.description="●" if color==BLACK else "○" if color==WHITE else "+"
            button.style.text_color="#141b20" if color else "#755729"
            button.style.font_size="31px" if color else "22px"
            button.style.button_color="#efcb8e" if action==latest else "#ddb77b"
            button.disabled=action not in legal
        turn="Black" if state.to_play==BLACK else "White"
        self.status.value=f"<b>Move {state.move_number} · {turn} to play</b> · {html.escape(self.session.result_text())}"
        self.pass_button.disabled=not human_turn or self.busy
        self.ai_button.disabled=state.terminal or self.busy or (mode!="watch" and state.to_play==human)
        self.undo_button.disabled=len(self.session.payload["timeline"])<=1 or self.busy
        self.new_button.disabled=self.busy;self.mode.disabled=self.busy;self.budget.disabled=self.busy
        self.watch_button.disabled=mode!="watch" or state.terminal or self.busy
        self.pause_button.disabled=self.paused

    def pause(self):
        self.paused=True;self.refresh()

    def _call(self, function):
        self.busy=True;self.refresh()
        try:
            result=function();self.message.value="<span style='color:#3b6556'>Saved.</span>";return result
        except Exception as error:
            self.message.value=f"<span style='color:#a53328'>{html.escape(str(error))}</span>";return None
        finally:self.busy=False;self.refresh()

    def _human(self, action):
        self.pause()
        result=self._call(lambda:self.session.play_vertex(gtp_action_to_vertex(action,9)))
        if result is not None and not self.session.state.terminal:self._single_ai()

    def _single_ai(self):
        result=self._call(self.session.ai_move)
        if result is not None:
            fallback=" · fallback pass" if result.get("fallback_pass") else ""
            if result.get("deadline_missed"): fallback += " · budget exceeded"
            self.message.value=f"AHR played {gtp_action_to_vertex(result['action'],9)} in {result.get('interaction_elapsed_s',result.get('elapsed_s',0)):.2f}s including save{fallback}. Strategic result: UNKNOWN. Saved."

    def _change(self,function):
        self.pause();self._call(function)

    def _settings(self,change):
        if self._sync:return
        self.pause();result=self._call(lambda:self.session.set_settings(self.mode.value,float(self.budget.value)))
        if result is None:
            self._sync=True
            try:self.mode.value=self.session.payload["mode"];self.budget.value=self.session.payload["time_budget_s"]
            finally:self._sync=False

    async def watch_batch(self,count):
        if self.session.payload["mode"]!="watch":raise ValueError("Select Watch AHR vs AHR first")
        if type(count) is not int or not 1<=count<=100:raise ValueError("Watch batches contain 1–100 moves")
        self.paused=False;self.refresh();completed=0
        try:
            for _ in range(count):
                if self.paused or self.session.state.terminal:break
                before=self.session.state.move_number;self._single_ai()
                if self.session.state.move_number==before:break
                completed+=1
                await asyncio.sleep(.05)
        finally:
            self.paused=True;self.refresh()
        return {"moves":completed,"terminal":self.session.state.terminal,"result":self.session.result_text()}

    def _start_watch(self):
        if self.task is None or self.task.done():self.task=asyncio.create_task(self.watch_batch(int(self.watch_count.value)))


## 4. Functional and recovery checks

The bounded checks below use the selected real checkpoint on CPU. They cover exact full-history restoration, legal AI moves, occupied-point rejection without mutation, undo, two-pass scoring, new-game reset, concurrent/stale-writer refusal, deliberate corruption rejection, and the widget callbacks. A disposable test session is separate from your saved playable game.

In [ ]:
MODEL_RUN12 = os.environ.get("AHR12_RUN_ID", os.environ.get("AHR12_MODEL_RUN", "go09-first-selfplay-v1"))
SELECTION12 = os.environ.get("AHR12_SELECTION", "latest")
DEVICE12 = os.environ.get("AHR12_DEVICE", "cpu")
SESSION12 = os.environ.get("AHR12_SESSION_ID", "play12-v2-main")
SEARCH12 = {"max_depth": 2, "branch_width": 4, "max_nodes": 32, "queen_cycles": 1, "temperature": 1.0}
print({"model": MODEL_RUN12, "selection": SELECTION12, "session": SESSION12, "device": DEVICE12})
# A unique validation session avoids resetting any existing user game.
VALIDATION_ID12 = "play12-validation-" + uuid.uuid4().hex[:12]
TEST12 = PlaySession12(PROJECT, VALIDATION_ID12, MODEL_RUN12, SELECTION12, "cpu", SEARCH12)
CHECKS12 = {}
TEST12.play_vertex("D4")
prior = copy.deepcopy(TEST12.payload)
try: TEST12.move(gtp_vertex_to_action("D4",9))
except IllegalMove: pass
else: raise AssertionError("Occupied point accepted")
assert TEST12.payload == prior
CHECKS12["occupied_move_transactional"] = True
SEARCH_RESULT12 = TEST12.ai_move()
assert TEST12.state.move_number == 2 and len(TEST12.state.history) >= 2
CHECKS12["real_checkpoint_ai_move"] = {"action":gtp_action_to_vertex(SEARCH_RESULT12["action"],9),"status":SEARCH_RESULT12["status"],"elapsed_s":SEARCH_RESULT12["elapsed_s"]}
SAVED12 = copy.deepcopy(TEST12.payload)
EXPECTED_RNG12 = (random.random(), float(np.random.random()), torch.rand(3).tolist())
RESTORED12 = PlaySession12(PROJECT, VALIDATION_ID12, MODEL_RUN12, SELECTION12, "cpu", SEARCH12)
assert RESTORED12.resumed and RESTORED12.payload == json.loads(play12_json(SAVED12))
assert (random.random(), float(np.random.random()), torch.rand(3).tolist()) == EXPECTED_RNG12
assert RESTORED12.state.to_dict() == TEST12.state.to_dict()
CHECKS12["exact_state_history_rng_resume"] = True
RESTORED12.undo()
assert RESTORED12.state.to_dict() == SAVED12["timeline"][-2]["state"]
try: TEST12.undo()
except RuntimeError as error: assert "stale" in str(error)
else: raise AssertionError("Stale writer accepted")
CHECKS12["undo_and_stale_writer"] = True
with play12_lock(RESTORED12.root):
    try: RESTORED12.set_settings(mode="watch")
    except RuntimeError as error: assert "writer" in str(error)
    else: raise AssertionError("Concurrent writer accepted")
CHECKS12["exclusive_writer"] = True
RESTORED12.move(81);RESTORED12.move(81)
assert RESTORED12.state.terminal and "wins" in RESTORED12.result_text()
CHECKS12["two_pass_terminal_result"] = True
RESTORED12.new_game();assert RESTORED12.state.to_dict()==initial_state(9,7.5).to_dict()
CHECKS12["new_game_exact_reset"] = True
# Corruption is injected only in this disposable session and restored in finally.
manifest12 = json.loads((RESTORED12.root/"manifest.json").read_text())
corrupt12 = RESTORED12.root/manifest12["latest"];original12=corrupt12.read_bytes()
try:
    atomic_write_bytes(corrupt12,b"{}")
    try: play12_read(RESTORED12.root,PROJECT)
    except ValueError as error: assert "Corrupt" in str(error)
    else: raise AssertionError("Corrupt game accepted")
finally:atomic_write_bytes(corrupt12,original12)
CHECKS12["corruption_refused"] = True
UI_TEST12=PlayWidget12(RESTORED12)
UI_TEST12.points[gtp_vertex_to_action("C3",9)].click()
assert RESTORED12.state.move_number==2
UI_TEST12.undo_button.click();assert RESTORED12.state.move_number==1
UI_TEST12.new_button.click();assert RESTORED12.state.move_number==0
UI_TEST12.mode.value="watch"
WATCH_CHECK12=await UI_TEST12.watch_batch(2)
assert WATCH_CHECK12["moves"]==2 and RESTORED12.state.move_number==2 and not RESTORED12.state.terminal
assert UI_TEST12.paused and UI_TEST12.watch_button.disabled is False
CHECKS12["widget_move_undo_new_watch_callbacks"] = True
CHECKS12["watch_cap_is_pause_not_result"] = WATCH_CHECK12
UI_TEST12.widget.close()
assert go_model_digest(RESTORED12.model)==RESTORED12.payload["model_metadata"]["model_sha256"]
assert len(json.loads((RESTORED12.root/"manifest.json").read_text())["entries"])==2
print({"status":"PASS","checks":list(CHECKS12),"AI_move":CHECKS12["real_checkpoint_ai_move"]})

## 5. Play or watch

Run this cell to open or resume your game. A live kernel is required for buttons. Click an empty point when it is your turn. To watch, select **Watch AHR vs AHR**, choose the batch size and click **Watch batch**. Nothing starts automatically. Pause can be used between moves; interrupting the kernel leaves the latest committed position intact.

Your model is pinned for the life of this session. `SESSION12` identifies the saved game; `MODEL_RUN12` and `SELECTION12` identify its starting checkpoint. To compare or play a newly trained model, use a different session ID.

In [ ]:
GAME12 = PlaySession12(PROJECT, SESSION12, MODEL_RUN12, SELECTION12, DEVICE12, SEARCH12)
APP12 = PlayWidget12(GAME12)
display(APP12.widget)
print({"session": SESSION12, "resumed": GAME12.resumed, "saved_move": GAME12.state.move_number,
       "checkpoint": str((GAME12.root/"manifest.json").relative_to(PROJECT)),
       "model_step": GAME12.payload["model_metadata"]["optimizer_step"], "automatic_game_started": False})

## 6. Portable board and validation receipt

The figure is a static view of the validation game, not a screenshot of the live widget controls. It uses the same exact state and coordinates. The live controls above continue to update during play. Browser-specific widget rendering must be checked in your notebook viewer; the functional callback checks are recorded separately.

In [ ]:
FIGURE12=play12_board_figure(RESTORED12,"AHR Go · validated board view")
SNAPSHOT12=PROJECT/"reports/12_play_board_preview.png"
FIGURE12.savefig(SNAPSHOT12,dpi=150,facecolor=FIGURE12.get_facecolor())
display(FIGURE12);plt.close(FIGURE12)
BOOK12=nbformat.read(PROJECT/"notebooks/12_play_and_watch.ipynb",4);nbformat.validate(BOOK12)
REPORT12={"schema":"ahr12-interface-validation-v1","status":"PASS","checks":CHECKS12,
    "source_hashes":play12_sources(PROJECT),"notebook":"notebooks/12_play_and_watch.ipynb",
    "source_sha256":hashlib.sha256("\n".join(c.source for c in BOOK12.cells if c.cell_type=="code").encode()).hexdigest(),
    "selected_model":RESTORED12.payload["model_metadata"],"device":"cpu","training_started":False,
    "default_game_started":False,"strength":"NOT_EVALUATED","validation_session":VALIDATION_ID12,
    "play_session":SESSION12,"snapshot":str(SNAPSHOT12.relative_to(PROJECT)),
    "widget_callbacks":"PASS","board_image_review":"PENDING_EXTERNAL_VISUAL_INSPECTION",
    "widget_browser_layout_review":"NOT_PERFORMED_NO_BROWSER_VIEWER",
    "resume_cell":PLAY12_RESUME,"runtime_dependencies":{"ipywidgets":widgets.__version__}}
atomic_json(PROJECT/"reports/12_play_validation.json",REPORT12)
print({"status":"PASS","checks":len(CHECKS12),"receipt":"reports/12_play_validation.json","strength":"NOT_EVALUATED"})